# Python API Example - End-of-Day JKM-TTF vs Arb Breakevens

Here we plot JKM-TTF prices against relevant Arb Breakeven levels. This script can also be used for TTF prices vs TTF breakevens.

For a full explanation of how to import our Intraday Contracts or Cargo Arb Breakevens data, please refer to our Python Jupyter Notebook Code Samples:

https://www.sparkcommodities.com/api/code-examples/jupyter.html

### Have any questions?

If you have any questions regarding our API, or need help accessing specific datasets, please contact us at:

__data@sparkcommodities.com__

or refer to our API website for more information about this endpoint:
https://www.sparkcommodities.com/api/

## 1. Importing Data

Here we define the functions that allow us to retrieve the valid credentials to access the Spark API.

This section can remain unchanged for most Spark API users.

In [ ]:
import json
import os
import sys
import pandas as pd
import numpy as np
from base64 import b64encode
from urllib.parse import urljoin
from io import StringIO
import datetime

try:
    from urllib import request, parse
    from urllib.error import HTTPError
except ImportError:
    raise RuntimeError("Python 3 required")

In [ ]:
API_BASE_URL = "https://api.sparkcommodities.com"


def retrieve_credentials(file_path=None):
    """
    Find credentials either by reading the client_credentials file or reading
    environment variables
    """
    if file_path is None:
        client_id = os.getenv("SPARK_CLIENT_ID")
        client_secret = os.getenv("SPARK_CLIENT_SECRET")
        if not client_id or not client_secret:
            raise RuntimeError(
                "SPARK_CLIENT_ID and SPARK_CLIENT_SECRET environment vars required"
            )
    else:
        # Parse the file
        if not os.path.isfile(file_path):
            raise RuntimeError("The file {} doesn't exist".format(file_path))

        with open(file_path) as fp:
            lines = [l.replace("\n", "") for l in fp.readlines()]

        if lines[0] in ("clientId,clientSecret", "client_id,client_secret"):
            client_id, client_secret = lines[1].split(",")
        else:
            print("First line read: '{}'".format(lines[0]))
            raise RuntimeError(
                "The specified file {} doesn't look like to be a Spark API client "
                "credentials file".format(file_path)
            )

    print(">>>> Found credentials!")
    
    return client_id, client_secret


def do_api_post_query(uri, body, headers):
    """
    OAuth2 authentication requires a POST request with client credentials before accessing the API.
    This POST request will return an Access Token which will be used for the API GET request.
    """
    url = urljoin(API_BASE_URL, uri)

    data = json.dumps(body).encode("utf-8")

    # HTTP POST request
    req = request.Request(url, data=data, headers=headers)
    try:
        response = request.urlopen(req)
    except HTTPError as e:
        print("HTTP Error: ", e.code)
        print(e.read())
        sys.exit(1)

    resp_content = response.read()

    # The server must return HTTP 201. Raise an error if this is not the case
    assert response.status == 201, resp_content

    # The server returned a JSON response
    content = json.loads(resp_content)

    return content


def do_api_get_query(uri, access_token, format='json'):
    """
    After receiving an Access Token, we can request information from the API.
    Supports both JSON (default) and CSV responses.
    """
    url = urljoin(API_BASE_URL, uri)

    if format == 'json':
        headers = {
            "Authorization": "Bearer {}".format(access_token),
            "Accept": "application/json",
        }
    elif format == 'csv':
        headers = {
            "Authorization": "Bearer {}".format(access_token),
            "Accept": "text/csv",
        }
    else:
        raise ValueError("The format parameter only takes 'csv' or 'json' as inputs")

    print("Fetching {}".format(url))

    # HTTP GET request
    req = request.Request(url, headers=headers)
    try:
        response = request.urlopen(req)
    except HTTPError as e:
        print("HTTP Error: ", e.code)
        print(e.read())
        sys.exit(1)

    resp_content = response.read()

    # The server must return HTTP 200. Raise an error if this is not the case
    assert response.status == 200, resp_content

    # Storing response based on requested format
    if format == 'json':
        content = json.loads(resp_content)
    elif format == 'csv':
        content = resp_content

    return content


def get_access_token(client_id, client_secret):

    payload = "{}:{}".format(client_id, client_secret).encode()
    headers = {
        "Authorization": b64encode(payload).decode(),
        "Accept": "application/json",
        "Content-Type": "application/json",
    }
    body = {
        "grantType": "clientCredentials",
    }

    content = do_api_post_query(uri="/oauth/token/", body=body, headers=headers)

    print(
        ">>>> Successfully fetched an access token {}****, valid {} seconds.".format(
            content["accessToken"][:5], content["expiresIn"]
        )
    )

    return content["accessToken"]

## N.B. Credentials

Here we call the above functions, and input the file path to our credentials.

N.B. You must have downloaded your client credentials CSV file before proceeding. Please refer to the API documentation if you have not dowloaded them already.

The code then prints the available prices that are callable from the API, and their corresponding Python ticker names are displayed as a list at the bottom of the Output.

In [ ]:
# Insert file path to your client credentials here
client_id, client_secret = retrieve_credentials(file_path="/tmp/client_credentials.csv")

# Authenticate:
access_token = get_access_token(client_id, client_secret)

# 2. Arb Breakevens Reference Data

Here we use the Cargo Netbacks Reference Data endpoint to return the UUID codes associated with each FoB Port available on the Arb Breakevens endpoint. 

In [ ]:
# Define the function for listing all netbacks
def list_netbacks(access_token):

    content = do_api_get_query(
        uri="/v1.0/netbacks/reference-data/", access_token=access_token
    )

    # retrieve release dates separately
    reldates = content["data"]["staticData"]["sparkReleases"]

    # return reference data
    refdata_dict = content["data"]

    return reldates, refdata_dict

# Fetch reference data:
reldates, refdata_dict = list_netbacks(access_token)

# formatting as a Dataframe
available_df = pd.json_normalize(refdata_dict['staticData']['fobPorts'])
available_df

# 3. Endpoint Fetch Functions

## 3.1 Cargo Arb Breakevens

Details on these parameters can be found on the API docs website:

https://www.sparkcommodities.com/api/lng-cargo/netbacks-arb-breakevens.html

In [ ]:
from io import StringIO

def fetch_breakevens(access_token, ticker, via=None, breakeven='jkm-ttf', start=None, end=None, format='json'):


    query_params = breakeven + '/' + "?fob-port={}".format(ticker)

    if via is not None:
        query_params += "&via-point={}".format(via)
    if start is not None:
        query_params += "&start={}".format(start)
    if end is not None:
        query_params += "&end={}".format(end)

    uri = "/v1.0/netbacks/arb-breakevens/{}".format(query_params)
    print(uri)
    content = do_api_get_query(
        uri="/v1.0/netbacks/arb-breakevens/{}".format(query_params),
        access_token=access_token, format=format,
    )
    
    if format == 'json':
        data = content['data']
    else:
        data = content.decode('utf-8')
        data = pd.read_csv(StringIO(data))
        data['ReleaseDate'] = pd.to_datetime(data['ReleaseDate'])

    return data


## 3.2 Intraday EOD - Contracts Data (JKM-TTF)

Details on these parameters can be found on our API docs website:

https://www.sparkcommodities.com/api/intraday/eod.html

In [ ]:
## Defining the function

def fetch_historical_contracts(access_token, contract, unit, start, end, format='json'):
    
    query_params = "?contract={}".format(contract)
    query_params += "&unit={}".format(unit)

    query_params += "&start={}".format(start)
    query_params += "&end={}".format(end)

    uri="/v1.0/intraday/contracts/eod/{}".format(query_params)
    print(uri)
    
    content = do_api_get_query(
        uri=uri, access_token=access_token, format=format
    )

    if format == 'json':
        data = content
    elif format == 'csv':
        # if there's no data to show, returns raw response (empty string) and "No Data to Show" message
        if len(content) == 0:
            data = pd.DataFrame()
            print('No Data to Show')
        else:
            data = content.decode('utf-8')
            data = pd.read_csv(StringIO(data))  # automatically converting into a Pandas DataFrame when choosing CSV format

    return data


# 4. Historical Data Function

Currently, a maximum of 1 year of historical data can be called at one time. If more data points are required, the below code can be used, which splits the requested date range into looped calls of 365 historical days at a time.

In [ ]:
import datetime

def loop_historical_jkmttf(access_token, contract=None, breakeven_type=None, ticker=None, via=None, percent_hire=None, unit=None, hist_start=None, hist_end=None):
    
    hist_diff = (datetime.datetime.strptime(hist_end, '%Y-%m-%d') - datetime.datetime.strptime(hist_start, '%Y-%m-%d')).days
    t = 0

    starts = []
    ends = []

    w = 365
    print(hist_diff)
    while t < hist_diff:
        print(t)
        # initialising dataframe
        if t == 0 and hist_diff>w:
            diff_end = datetime.datetime.strftime(datetime.datetime.strptime(hist_start, '%Y-%m-%d') + pd.Timedelta(days=w), '%Y-%m-%d')
            hist_df = fetch_historical_contracts(access_token, contract=contract, unit=unit, start=hist_start, end=diff_end, format='csv')

            starts.append(hist_start)
            ends.append(diff_end)
        
        elif t==0 and hist_diff<=w:
            hist_df = fetch_historical_contracts(access_token, contract=contract, unit=unit, start=hist_start, end=hist_end, format='csv')

        # appending additional historical data
        else:
            if t < hist_diff-w:
                diff_start = datetime.datetime.strftime(datetime.datetime.strptime(hist_start, '%Y-%m-%d') + pd.Timedelta(days=t+1), '%Y-%m-%d')
                diff_end = datetime.datetime.strftime(datetime.datetime.strptime(diff_start, '%Y-%m-%d') + pd.Timedelta(days=w), '%Y-%m-%d')
                historical_addition = fetch_historical_contracts(access_token, contract=contract, unit=unit, start=diff_start, end=diff_end, format='csv')
                try:
                    hist_df = pd.concat([hist_df,historical_addition])
                #exception if hist_df is empty
                except:
                    hist_df = historical_addition.copy()
                starts.append(hist_start)
                ends.append(diff_end)
            else:
                diff_start = datetime.datetime.strftime(datetime.datetime.strptime(hist_start, '%Y-%m-%d') + pd.Timedelta(days=t+1), '%Y-%m-%d')
                diff_end = datetime.datetime.strftime(datetime.datetime.strptime(diff_start, '%Y-%m-%d') + pd.Timedelta(days=hist_diff-t), '%Y-%m-%d')
                historical_addition = fetch_historical_contracts(access_token, contract=contract, unit=unit, start=diff_start, end=diff_end, format='csv')
                hist_df = pd.concat([hist_df, historical_addition])
                starts.append(hist_start)
                ends.append(diff_end)
        
        #looping by year
        t += w

    for c in list(hist_df.columns)[13:]:
        hist_df[c] = pd.to_numeric(hist_df[c])
    hist_df['ReleaseDate'] = pd.to_datetime(hist_df['ReleaseDate'])

    return hist_df

# 5. Calling the Historical Data

In [ ]:
# Calling the extended historical data function

start = '2026-01-01'
today = datetime.date.today().strftime('%Y-%m-%d')

tsab = available_df[available_df["name"] == 'Sabine Pass']["uuid"].iloc[0]

# Sabine Pass (NEA via COGH) Arb Breakeven
sabcogh_df = fetch_breakevens(access_token, tsab, via='cogh', breakeven='jkm-ttf', format='csv')

# Sabine Pass Arb (NEA via Panama Canal) Breakeven
sabpan_df = fetch_breakevens(access_token, tsab, via='panama', breakeven='jkm-ttf', format='csv')

# JKM-TTF prices
jkmttf = loop_historical_jkmttf(access_token, contract='jkm-ttf', unit='usd-per-mmbtu', hist_start=start, hist_end=today)

sabcogh_df.head(5)


In [ ]:
# Combining the contract data & the breakevens data into a single dataframe

# listing all breakevens Dataframes in a list
break_dataframes = [sabcogh_df, sabpan_df]
break_names = ['Sabine Pass (COGH)', 'Sabine Pass (Panama)']

# initialising dataframe with the contract data
combined_df = jkmttf.copy()
combined_df = combined_df.rename(columns={'Value':'ContractValue'})

# merging with the breakevens dataframes
dc = 0
for df in break_dataframes:
    print(break_names[dc])
    df = df.rename(columns={'JKMTTFSpreadBreakeven':break_names[dc]})
    combined_df = pd.merge(combined_df, df[['ReleaseDate', 'DeliveryMonthIndex', break_names[dc]]], 
                            left_on=['ReleaseDate', 'PeriodIndex'], right_on=['ReleaseDate', 'DeliveryMonthIndex'], how='inner')
    dc+=1

combined_df = combined_df[['ReleaseDate', 'PeriodType', 'PeriodFrom','PeriodName','PeriodIndex', 'Unit', 'ContractValue'] + break_names]
combined_df['PeriodFrom'] = pd.to_datetime(combined_df['PeriodFrom'])

# Filtering out quarters, seasons and Cal data where arb breakeven levels are not available
final_df = combined_df[combined_df['PeriodType'] == 'month'].copy()

# creating columns to show the difference between the Contract price and the Breakeven levels
final_df['Sabine Pass (COGH) - Breakeven Diff'] = final_df['ContractValue'] - final_df['Sabine Pass (COGH)']
final_df['Sabine Pass (Panama) - Breakeven Diff'] = final_df['ContractValue'] - final_df['Sabine Pass (Panama)']

In [ ]:
final_df.head(5)

# 6. Plotting

## 6.1. Plotting Latest Forward Curve

In [ ]:
# create dataframe by filtering combined_df
latest_df =  final_df[final_df['ReleaseDate'] == final_df['ReleaseDate'].max()]

latest_df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

fig, ax = plt.subplots(figsize=(14,6))

# Plotting a thicker y=0 axis line
ax.hlines(0, latest_df['PeriodFrom'].iloc[-1] + pd.Timedelta(days=10), 
            latest_df['PeriodFrom'].iloc[0] - pd.Timedelta(days=10), color='grey', alpha=0.85, zorder=0)

# Plotting JKM-TTF forward Curve & Arb Breakevens
ax.plot(latest_df['PeriodFrom'], latest_df['ContractValue'], color='#dc853f', label = 'JKM-TTF')
ax.plot(latest_df['PeriodFrom'], latest_df['Sabine Pass (COGH)'], color='#008bcc', label='Sabine Pass (via COGH) Arb Breakeven')
ax.plot(latest_df['PeriodFrom'], latest_df['Sabine Pass (Panama)'], color='#008e74', label='Sabine Pass (via Panama) Arb Breakeven')

# shading green when the US arb is open (JKM-TTF > Sabine Pass Breakeven)
ax.fill_between(latest_df['PeriodFrom'], latest_df['ContractValue'], latest_df['Sabine Pass (COGH)'], 
                where=(latest_df['Sabine Pass (COGH) - Breakeven Diff'] > 0), color='mediumseagreen', alpha=0.2)

# shading red when the US Arb (COGH) is closed (JKM-TTF < US Arb (COGH) Breakeven)
ax.fill_between(latest_df['PeriodFrom'], latest_df['Sabine Pass (Panama)'], latest_df['ContractValue'], 
                where=(latest_df['Sabine Pass (Panama) - Breakeven Diff'] < 0), color='red', alpha=0.1)

plt.xlim([latest_df['PeriodFrom'].iloc[0]-pd.Timedelta(days=10), latest_df['PeriodFrom'].iloc[-1]+pd.Timedelta(days=10)])

plt.legend(loc='lower left')

## 6.2 Plotting Historical Evolution of a Contract Month

In [ ]:
month = 'Dec26'

month_df = final_df[final_df['PeriodName'] == month]

In [ ]:
fig, ax2 = plt.subplots(figsize=(14,6))

# Plotting a thicker y=0 axis line
ax2.hlines(0, month_df['ReleaseDate'].iloc[-1] + pd.Timedelta(days=10), 
                month_df['ReleaseDate'].iloc[0] - pd.Timedelta(days=10), color='grey', alpha=0.85, zorder=0)

# Plotting JKM-TTF forward Curve & Arb Breakevens
ax2.plot(month_df['ReleaseDate'], month_df['ContractValue'], color='#dc853f', label = 'JKM-TTF')
ax2.plot(month_df['ReleaseDate'], month_df['Sabine Pass (COGH)'], color='#008bcc', label='Sabine Pass (via COGH) Arb Breakeven')
ax2.plot(month_df['ReleaseDate'], month_df['Sabine Pass (Panama)'], color='#008e74', label='Sabine Pass (via Panama) Arb Breakeven')

# shading green when the US arb is open (JKM-TTF > Sabine Pass Breakeven)
ax2.fill_between(month_df['ReleaseDate'], month_df['ContractValue'], month_df['Sabine Pass (COGH)'], 
                where=(month_df['Sabine Pass (COGH) - Breakeven Diff'] > 0), color='mediumseagreen', alpha=0.2)

# shading red when the US Arb (COGH) is closed (JKM-TTF < US Arb (COGH) Breakeven)
ax2.fill_between(month_df['ReleaseDate'], month_df['Sabine Pass (Panama)'], month_df['ContractValue'], 
                where=(month_df['Sabine Pass (Panama) - Breakeven Diff'] < 0), color='red', alpha=0.1)

plt.xlim([month_df['ReleaseDate'].iloc[0]-pd.Timedelta(days=10), month_df['ReleaseDate'].iloc[-1]+pd.Timedelta(days=10)])

plt.legend(loc='upper left')